# Sauda shopping assistant: free testing

Click the **key icon 🔑** on the left (Secrets) and add any free keys you have: `GROQ_API_KEY`, `GEMINI_API_KEY`, `CEREBRAS_API_KEY`, `OPENROUTER_API_KEY`, `MISTRAL_API_KEY`. Turn on **Notebook access** for each. Free plans have per-minute and daily limits; the app moves to the next provider when one is busy, and answers simple product lists itself without using any quota.

Run the cells from top to bottom: press ▶ on each one and wait for it to finish.

**Got an updated version from Claude?** Use **Runtime → Restart session**, then run the cells again from Step 1.

## Step 1: Download and install

In [ ]:
%cd /content
!rm -rf /content/Usman-hassan
!git clone -q --depth 1 -b claude/ecommerce-ai-assistant-t27kqw https://github.com/Usmanhassan7654/Usman-hassan.git
%cd /content/Usman-hassan/sauda-assistant
!pip install -q -r requirements.txt
print("Installed.")

## Step 2: Load your keys

In [ ]:
import os
from google.colab import userdata

KEYS = {"groq": "GROQ_API_KEY", "gemini": "GEMINI_API_KEY", "cerebras": "CEREBRAS_API_KEY",
        "openrouter": "OPENROUTER_API_KEY", "mistral": "MISTRAL_API_KEY"}
for name in KEYS.values():
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value
print("Keys loaded:", ", ".join(p for p, k in KEYS.items() if os.environ.get(k)) or "none")

## Step 3: Check the app itself (no keys needed)
Should end with **passed**.

In [ ]:
!python -m pytest -q 2>&1 | tail -3

## Step 4: Check your providers
Tries each provider you have a key for and prints **OK** or the exact error.

In [ ]:
# Model names (change if a provider says a model isn't found; the check lists the ones your key can use).
MODELS = {"groq": "openai/gpt-oss-120b", "cerebras": "gpt-oss-120b",
          "gemini": "gemini-flash-latest", "openrouter": "openai/gpt-oss-20b:free", "mistral": "mistral-small-latest"}

import sys, json, shutil
sys.path.insert(0, ".")
for name in [m for m in sys.modules if m == "sauda" or m.startswith("sauda.")]:
    del sys.modules[name]          # always use the freshly downloaded code
from sauda import llm
from sauda.tools import TOOLS

def available():
    return [p for p in ("groq", "cerebras", "gemini", "openrouter", "mistral") if os.environ.get(KEYS[p])]

def check(provider):
    m = llm.build(f"{provider}:{MODELS[provider]}")
    convo = [{"role": "user", "content": "I need milk. Record it as a need."}]
    try:
        r1 = m.complete("You are a shopping assistant. Use the update_needs tool.", convo, TOOLS)
        line = f"{m.name}  OK  (tool calls: {[c['name'] for c in r1.tool_calls] or 'none'})"
        if r1.tool_calls:
            convo += [{"role": "assistant", "content": r1.text, "tool_calls": r1.tool_calls}]
            convo += [{"role": "tool", "id": c["id"], "name": c["name"], "content": json.dumps({"ok": True})} for c in r1.tool_calls]
            m.complete("You are a shopping assistant.", convo, TOOLS)
            line += ", follow-up OK"
        print(line)
        return True
    except Exception as e:
        print(f"{m.name}  FAILED: {e}")
        try:
            print("   Models it can use:", ", ".join(m.list_models()[:40]))
        except Exception:
            pass
        return False

PROVIDERS = [p for p in available() if check(p)]
SPEC = ",".join(f"{p}:{MODELS[p]}" for p in PROVIDERS)
print("\nWill use, in this order:", SPEC or "no provider works yet. Add a key in Secrets and run Step 2 again.")

## Step 5: Chat with the assistant
Change the message in the last line and run the cell again. Simple product lists ("doodh aur double roti") are answered by the app itself without using any API quota.

In [ ]:
import uuid
from sauda import ingest
from sauda.db import Repo
from sauda.agent import Agent

repo = Repo("colab.db")
scope = repo.add_store("demo-mart", "Demo Mart", "demo-khi", "Karachi", api_key="demo-store-key")
if not repo.catalog_stats(scope)["published_products"]:
    ingest.sync(repo, scope, "data/sample_pk_grocery.csv")
agent = Agent(repo, model=llm.build(SPEC) if SPEC else None, fast_path=True)
session = uuid.uuid4().hex

def ask(message):
    r = agent.handle(scope, session, message, emit=lambda e: print("  …", e["text"]))
    print("\nAssistant:", r.reply, "\n")
    for card in r.cards:
        if card["type"] == "plan":
            print("Suggested list:")
            for l in card["lines"]:
                print(f"  {l['packs']} x {l['name']}  Rs {l['line_total']:,.0f}")
            print(f"  Total: Rs {card['total']:,.0f}")
            for b in card["budgets"]:
                print(f"  Budget {b['budget']:,.0f}: {'OVER by ' + format(-b['remaining'], ',.0f') if b['over'] else format(b['remaining'], ',.0f') + ' left'}")
        elif card["type"] == "options":
            for g in card["groups"]:
                print(g["label"] + ":")
                for o in g["options"]:
                    print(f"  {o['ref']}. {o['name']}  Rs {o['price']:,.0f}")
        elif card["type"] == "comparison":
            for i in card["items"]:
                print(f"  {i['ref']}. {i['name']}  Rs {i['price']:,.0f}  {i['unit_price']}")
    if r.basket["items"]:
        print(f"Basket: {r.basket['item_count']} items, Rs {r.basket['total']:,.0f}")
    if r.grounding.get("corrected"):
        print("(The assistant corrected its first reply:", r.grounding["corrected"], ")")
    how = "answered by the app itself, no API used" if r.usage.get("fast_path") else f"{r.usage['model_calls']} model calls via {', '.join(r.usage['models']) or 'none'}"
    print(f"[{how}]")
    for err in r.usage.get("errors", []):
        print("PROVIDER ERROR (send this to Claude):", err)

ask("doodh aur double roti")

In [ ]:
ask("haan daal do")

In [ ]:
ask("chicken biryani banana hai 6 logon ke liye")

## Step 6: Run the automatic customer tests
Progress is printed after every answer. **QUICK = True** runs 4 core tests; set it to False for all 10. On free plans each answer can take up to a minute.

Copy what this cell prints and send it to Claude.

In [ ]:
QUICK = True

if not SPEC:
    print("Nothing to test with yet: add a key in Secrets, then run Steps 2 and 4 again.")
else:
    flags = "--quick" if QUICK else ""
    print("Using:", SPEC)
    !python -u evals/journeys.py --llm "{SPEC}" {flags}

## Step 7 (optional): Open the full chat website
Run this, then click the link that appears.

In [ ]:
import subprocess, time
from google.colab import output
env = {**os.environ, "SAUDA_DEMO": "1", "SAUDA_DB": "web.db", "SAUDA_LLM": SPEC}
server = subprocess.Popen(["python", "-m", "uvicorn", "sauda.server:app", "--port", "8000"], env=env,
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)
output.serve_kernel_port_as_window(8000, anchor_text="Open the Sauda chat website")